# M5 · 의미 검색 RAG 검증

09b의 의미 벡터를 실제 LCEL RAG에 연결한다. 업종·장비 필터와 작업 맥락·이전 질문은 검색에 반영하되, 답변 근거는 검색된 문서로 제한한다. 보고서에서 원인과 대책을 함께 물으면 해당 절의 페이지도 가져온다.

In [ ]:
from pathlib import Path
import json
from dotenv import dotenv_values
from langchain_openai import ChatOpenAI
from src.retriever import PersonalRetriever
from src.rag_chain import build_rag_chain
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists()),None)
assert ROOT is not None
config=dotenv_values(ROOT/'.env')
key=(config.get('OPENAI_API_KEY') or '').strip()
assert key,'OPENAI_API_KEY가 필요합니다.'
retriever=PersonalRetriever(ROOT,mode='semantic')
model=ChatOpenAI(model=config.get('OPENAI_MODEL') or 'gpt-6-luna',api_key=key,timeout=60,max_retries=0,max_completion_tokens=1200,reasoning_effort='low')
chain=build_rag_chain(retriever.retrieve,model,min_similarity=0.38)
ocr_retriever=PersonalRetriever(ROOT,mode='lexical')
ocr_chain=build_rag_chain(ocr_retriever.retrieve,model,min_similarity=0.12)
tbm_retriever=PersonalRetriever(ROOT,mode='lexical')
tbm_chain=build_rag_chain(tbm_retriever.retrieve_tbm,model,min_similarity=0.12)


## ① 사례·⑥ 보고서·⑧ 지침 질문

In [ ]:
cases=[
 {'id':'GUIDE','question':'이동식 사다리 작업 전에 무엇을 점검해야 하나요?','source_type':'kosha_guide','equipment':'사다리'},
 {'id':'OCR_GUIDE','question':'작업장 계단의 안전난간에서 상부난간대가 120cm 이하인 경우 중간난간대는 어디에 설치해야 하나요?',
  'source_type':'kosha_guide','equipment':'계단'},
 {'id':'TBM','question':'현재 작업에 맞춰 검색된 사고사례와 안전지침에 근거한 TBM 시작 전 체크리스트를 작성해 주세요.',
  'work_context':'건설현장 이동식 사다리 사용 전 상태·설치 점검','equipment':'사다리',
  'required_source_types':['sif','kosha_guide']},
 {'id':'SIF','question':'화학물질이 남은 드럼통을 산소절단하다 폭발한 사고의 재해유발요인과 감소대책은?','source_type':'sif','industry_major':'제조업','work_context':'제조업 사업장의 드럼 절단 작업'},
 {'id':'MOEL','question':'화성시 톤백 깔림 사고의 원인과 재해예방대책은?','source_type':'moel_report'},
]
records=[]
for case in cases:
    runner=tbm_chain if case['id']=='TBM' else (ocr_chain if case['id']=='OCR_GUIDE' else chain)
    result=runner.invoke(case)
    row={'id':case['id'],'question':case['question'],'status':result['status'],'answer':result['answer'],
         'sources':[{'doc_id':s['doc_id'],'source_type':s['metadata']['source_type'],'page':s['metadata']['page'],'url':s['metadata']['source_url'],
                  'ocr_review_required':bool(s['metadata'].get('ocr_review_required',False))} for s in result['sources']]}
    records.append(row)
    print(case['id'],'상태:',row['status'],'출처:',[(s['doc_id'],s['page']) for s in row['sources']])
    print('답변:',row['answer'])

## 자료 밖 질문의 거절 여부

In [ ]:
negative_questions=['화성 탐사 로켓의 연료 종류는?','모나리자를 그린 화가는 누구인가?','2022년 월드컵 우승팀은 어디인가?']
for i,question in enumerate(negative_questions,1):
    result=chain.invoke({'question':question})
    records.append({'id':f'NEG{i}','question':question,'status':result['status'],'answer':result['answer'],'sources':[]})
    print(question,'→',result['status'],result['answer'])
path=ROOT/'data'/'personal'/'evaluation'/'answer_spotcheck_multisource.json'
path.write_text(json.dumps(records,ensure_ascii=False,indent=2),encoding='utf-8')

## 의미검색 RAG 검증 결과

- 이동식 사다리 사전 점검 질문에서 KOSHA GUIDE `A-G-4-2025` PDF 7쪽과 OCR로 복구한 8쪽이 검색됐다. 답변에는 인용 번호와 공식 PDF 링크·쪽 정보를 연결했다.
- OCR 청크는 본문을 읽기 쉽게 만들 수 있지만 인식 결과가 원문과 달라질 수 있다. OCR 페이지를 인용하면 화면의 재확인 표시를 보고 원문을 열어 대조한다.
- 이 노트북의 점검은 특정 질문에 대한 동작 확인이다. M6 검색 점수와 동일한 평가가 아니고, 모든 작업 유형에서 안전 답변이 되는지를 증명하지 않는다.

### 사용자가 답을 검토하는 순서

- 답변의 각 인용 번호를 해당 문서·PDF 페이지와 연결해 확인한다.
- 답변 문장이 검색된 근거보다 강하거나 구체적인 경우, 근거에 없는 부분은 채택하지 않는다.
- 자료에 없으면 모른다고 답하는 동작을 유지한다. 현장 판단은 실제 작업 조건과 공식 안전기준으로 다시 확인한다.
